# 2. Requesting GPUs on NRP (Instructor: Daniel Diaz)

This session guides participants through the process of requesting GPUs and other compute resources on the National Research Platform (NRP). Attendees will be introduced to the NRP portal, where they can explore available hardware, including standard GPUs and the Qualcomm Cloud AI 100 Ultra SoCs. The tutorial demonstrates how to submit a resource request, select the appropriate GPU type for a given workload, and understand scheduling constraints and allocation quotas.

## GPU's ON Kubernetes

Lets start by running a simple GPU job on Kubernetes. We will use the `nvidia-smi` command to check the GPU's available on the node.

*(The YAML configuration has been saved to the `yamls/` directory. If a YAML uses a fixed resource name with `<username>`, replace it with your username to avoid collisions.)*

## Kubernetes and Core Concepts

Before diving into deploying workloads, let's establish a baseline understanding of the core technologies that power the National Research Platform:

### Containers and Images
- **Containers**: Lightweight, standalone, executable packages that include everything needed to run a piece of software, including the code, runtime, system tools, and libraries.
- **Container Images**: The read-only templates used to create containers. In Nautilus, we use various images like the `ghcr.io/quic/cloud_ai_inference_ubuntu24` environment for Qualcomm workflows.

### Kubernetes Fundamentals
Kubernetes is an open-source container orchestration engine for automating deployment, scaling, and management of containerized applications.
- **Pods**: The smallest deployable computing units in Kubernetes. A pod can contain one or more tightly coupled containers. When you launch a JupyterHub instance, you are running inside a Pod.
- **Jobs**: Workloads designed to reliably execute a task to completion (e.g., training a model once), as opposed to long-running services.
- **Persistent Volume Claims (PVCs)**: Kubernetes resources that request storage allocation. They ensure your data persists independently of the lifecycle of any single pod.

### Hardware Acceleration
Kubernetes requires specialized extensions to manage and assign non-CPU hardware.
- **GPUs in Kubernetes**: Workloads can explicitly request NVIDIA GPUs (e.g. `nvidia.com/gpu`) by specifying resource limits in their deployment manifests.
- **Device Plugins**: Software components that advertise specialized hardware resources to the Kubernetes scheduler. For example, Qualcomm Cloud AI devices are exposed through a device plugin natively mapping as `qualcomm.com/qaic`.

### Basic kubectl Usage
The Kubernetes command-line tool, `kubectl`, allows you to run commands against Kubernetes clusters. You will use commands like `kubectl apply -f <file.yaml>` to deploy resources, or `kubectl get pods` to inspect running states.


## GPU Provisioning on NRP

To run workloads on GPUs or Qualcomm AI100 devices, you must **request the right resources and use compatible images**. Below is a concise reference; see the [NRP GPU types guide](https://nrp.ai/documentation/userdocs/running/gpu-pods/#choosing-gpu-type) for more detail.

### NVIDIA GPUs

- **Resource request:** In your pod spec, set `resources.limits` and `resources.requests` with the GPU resource key. For a generic GPU use `nvidia.com/gpu: <count>`. For a specific product (e.g. A100, A10, L40, RTX A6000, V100) use the product-specific resource (e.g. `nvidia.com/a100`, `nvidia.com/rtxa6000`) if your cluster exposes them.
- **Node selector (optional):** To target a specific GPU type, add a `nodeSelector` or `affinity` that matches node labels. For example, NRP nodes may have `nvidia.com/gpu.product` (e.g. `NVIDIA-A10`, `NVIDIA-L40`, `Tesla-V100-SXM2-32GB`). Use this when you need a particular GPU model.
- **Runtime / image:** Use a CUDA-enabled image (e.g. `nvidia/cuda:12.0.0-base-ubuntu22.04` or NRP scientific images with CUDA). The node runs the NVIDIA device plugin and container runtime so the GPU is available inside the container.
- **Example (generic GPU):** See `yamls/gpu-pod.yaml` — it requests `nvidia.com/gpu: 1` and runs `nvidia-smi`. Replace `<username>` in the pod name if sharing the namespace.

### Qualcomm Cloud AI 100

- **Resource request:** Use `qualcomm.com/qaic: <count>` in `resources.limits` and `resources.requests`. Each unit typically corresponds to one SoC (e.g. one AI 100 Ultra card can expose multiple devices).
- **Node selector:** Nodes with Qualcomm devices are labeled; use `nodeAffinity` with `qualcomm.com/qaic: Exists` (or the label your cluster uses) so the pod is scheduled on a node that has the device plugin and hardware.
- **Runtime / image:** Use a Qualcomm-supported image such as `ghcr.io/quic/cloud_ai_inference_ubuntu24` (see Part 3 and Part 4 for inference and RAG examples). The container must have the Cloud AI SDK and drivers expected by the device plugin.
- **Example:** See `yamls/qaic-inference.yaml` in Part 3/4 for a full inference pod using `qualcomm.com/qaic: 1`.

The GPU pod YAML is in `yamls/gpu-pod.yaml`. It requests one NVIDIA GPU and runs `nvidia-smi`. **Replace `<username>` in the pod name** with your username, then create the pod:

```bash
kubectl apply -n nrp-training-k8s -f yamls/gpu-pod.yaml
```

In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/gpu-pod.yaml

## MPI On Kubernetes

First lets run a basic CPU MPI job on Kubernetes. We will use the MPI operator to run the job.
We will calculate the value of Pi using the Monte Carlo method. The code is written in C and is available in the `mpi-pi` directory.

*(The YAML configuration has been saved to the `yamls/` directory)*

Copy the above yaml to a file called `mpi-pi.yaml` and run the following command to create the job.

```bash
kubectl apply -f mpi-pi.yaml
```

In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/mpi-pi.yaml

Now let's run a multi-node GPU MPI job on Kubernetes. We will use the MPI operator to run the job.

*(The YAML configuration has been saved to the `yamls/` directory)*

Copy the above yaml to a file called `mpi-tensorflow.yaml` and run the following command to create the job.

```bash
kubectl apply -f mpi-tensorflow.yaml
```

In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/mpi-tensorflow.yaml

## End

**Please make sure you did not leave any running pods. Jobs and associated completed pods are OK.**